# EDA — Indian Weather (T024)

Notebook de **análise exploratória** para o marco M02 / história S02. Restrição da equipa: **PyArrow** + **Matplotlib** + **NumPy** (sem pandas nem scikit-learn).

## Amostra e reprodutibilidade

A primeira célula de código define `N_AMOSTRA` (linhas lidas do Parquet após seleção de colunas). **Gráficos e estatísticas descritas aqui referem-se a essa amostra**, não necessariamente ao ficheiro completo, salvo indicação em contrário.

## Documentação relacionada

- Alvo e pipeline: [preprocessamento.md](../docs/preprocessamento.md)
- Desequilíbrio de classes: [imbalance.md](../docs/imbalance.md)


In [1]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# Tamanho da amostra (linhas) após leitura com colunas selecionadas
N_AMOSTRA = 500_000


def col_to_numpy(col, *, dtype=None):
    """Compat: PyArrow Array/ChunkedArray -> numpy.

    Algumas versões não aceitam kwargs em `to_numpy()`. Para ChunkedArray,
    combinamos chunks antes para evitar surpresas.
    """

    if hasattr(col, "combine_chunks"):
        try:
            col = col.combine_chunks()
        except Exception:
            pass

    try:
        arr = col.to_numpy(zero_copy_only=False)
    except TypeError:
        arr = col.to_numpy()

    if dtype is not None:
        return np.asarray(arr, dtype=dtype)
    return np.asarray(arr)


def resolve_repo_and_fig_dir() -> tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    roots = [cwd]
    if cwd.name == "notebooks":
        roots.append(cwd.parent)
    for root in roots:
        if (root / "requirements.txt").exists():
            fig = root / "notebooks" / "figuras"
            fig.mkdir(parents=True, exist_ok=True)
            return root, fig
    fig = cwd / "figuras"
    fig.mkdir(parents=True, exist_ok=True)
    return cwd, fig


REPO_ROOT, FIG_DIR = resolve_repo_and_fig_dir()
PARQUET = REPO_ROOT / "data" / "Indian_Weather_Dataset.parquet"

os.environ.setdefault("MPLBACKEND", "Agg")

COLS_LEITURA = [
    "datetime",
    "rain_label",
    "temperature_C",
    "humidity_pct",
    "precip_mm",
    "cloud_cover_pct",
    "pressure_hPa",
    "dew_point_C",
    "wind_speed_ms",
    "solar_radiation_Wm2",
    "lat",
    "lon",
    "hour",
    "month",
]

if not PARQUET.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {PARQUET}. Coloque o dataset em data/ e execute a partir da raiz do repositório ou abra o Jupyter com cwd na raiz."
    )

table = pq.read_table(PARQUET, columns=COLS_LEITURA, use_threads=True)
n_full = table.num_rows
if table.num_rows > N_AMOSTRA:
    table = table.slice(0, N_AMOSTRA)

print("REPO_ROOT:", REPO_ROOT)
print("FIG_DIR:", FIG_DIR)
print("Linhas no ficheiro (aprox.):", n_full)
print("Linhas usadas neste notebook:", table.num_rows)
print(table.schema)


REPO_ROOT: /home/jovyan/work
FIG_DIR: /home/jovyan/work/notebooks/figuras
Linhas no ficheiro (aprox.): 46082160
Linhas usadas neste notebook: 500000
datetime: timestamp[ms]
rain_label: int64
temperature_C: double
humidity_pct: int64
precip_mm: double
cloud_cover_pct: int64
pressure_hPa: double
dew_point_C: double
wind_speed_ms: double
solar_radiation_Wm2: double
lat: double
lon: double
hour: int64
month: int64


## Distribuição do alvo (`rain_label`)

Histograma de frequências absolutas. Para interpretação do desequilíbrio e estratégias (class weights, oversampling no Spark, etc.), ver [imbalance.md](../docs/imbalance.md).


In [2]:
labels = table.column("rain_label")
vc = pc.value_counts(labels)
labs = col_to_numpy(vc.field(0))
cnts = col_to_numpy(vc.field(1))
total = float(cnts.sum())

fig, ax = plt.subplots(figsize=(6, 4))
xs = [str(x) for x in labs]
bars = ax.bar(xs, cnts, color="#4C72B0", edgecolor="#222222", linewidth=0.8)
ax.set_xlabel("rain_label")
ax.set_ylabel("Contagem (amostra)")
ax.set_title("Distribuição do alvo na amostra")
for rect, n in zip(bars, cnts):
    pct = 100.0 * float(n) / total
    ax.annotate(
        f"{int(n):,}\n({pct:.1f}%)",
        xy=(rect.get_x() + rect.get_width() / 2, rect.get_height()),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
    )
ax.margins(x=0.15)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_rain_label_distribuicao.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_rain_label_distribuicao.png")
for a, b in zip(labs, cnts):
    print(a, int(b))


Figura: /home/jovyan/work/notebooks/figuras/eda_rain_label_distribuicao.png
0 459562
1 40438


## Relações feature–alvo

Médias de variáveis numéricas por classe (`rain_label`) via `Table.group_by` do PyArrow, e **boxplot** de `precip_mm` por classe (distribuição condicional na amostra).


In [3]:
agg_cols = [
    ("temperature_C", "mean", "°C"),
    ("humidity_pct", "mean", "%"),
    ("cloud_cover_pct", "mean", "%"),
]
g = table.group_by("rain_label").aggregate([(c, a) for c, a, _ in agg_cols])
print(g)

labs_mean = g.column("rain_label").to_pylist()
x = np.arange(len(labs_mean))
width = 0.55

fig, axes = plt.subplots(3, 1, figsize=(7, 7.2), sharex=True)
for ax, (col, _, unit) in zip(axes, agg_cols):
    vals = col_to_numpy(g.column(f"{col}_mean"), dtype=np.float64)
    bars = ax.bar(x, vals, width, color="#4C72B0", edgecolor="#222222", linewidth=0.6)
    ax.set_ylabel(f"média ({unit})")
    ax.set_title(f"{col} por rain_label")
    for rect, v in zip(bars, vals):
        ax.annotate(
            f"{v:.2f}",
            xy=(rect.get_x() + rect.get_width() / 2, rect.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )
axes[-1].set_xticks(x)
axes[-1].set_xticklabels([str(v) for v in labs_mean])
axes[-1].set_xlabel("rain_label")
fig.suptitle("Médias por classe — uma escala por variável (amostra)", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_medias_num_por_rain_label.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_medias_num_por_rain_label.png")


pyarrow.Table
temperature_C_mean: double
humidity_pct_mean: double
cloud_cover_pct_mean: double
rain_label: int64
----
temperature_C_mean: [[26.25144398361899,25.662985310845972]]
humidity_pct_mean: [[65.09294066959409,89.1622978386666]]
cloud_cover_pct_mean: [[45.21409733615921,93.4283841930857]]
rain_label: [[0,1]]
Figura: /home/jovyan/work/notebooks/figuras/eda_medias_num_por_rain_label.png


## Distribuição do alvo (`temperature_C`)

Como vamos tratar **regressão**, o foco é entender a faixa típica de temperatura, caudas/outliers e possíveis recortes (hora/mês/local). Aqui olhamos a distribuição na amostra, com estatísticas descritivas e percentis.


In [4]:

t = col_to_numpy(table.column("temperature_C"), dtype=np.float64)

t_valid = t[~np.isnan(t)]

stats = {
    "n_total": int(t.shape[0]),
    "n_valid": int(t_valid.shape[0]),
    "n_nan": int(np.isnan(t).sum()),
    "min": float(np.nanmin(t)),
    "p01": float(np.nanpercentile(t_valid, 1)),
    "p05": float(np.nanpercentile(t_valid, 5)),
    "p25": float(np.nanpercentile(t_valid, 25)),
    "median": float(np.nanpercentile(t_valid, 50)),
    "p75": float(np.nanpercentile(t_valid, 75)),
    "p95": float(np.nanpercentile(t_valid, 95)),
    "p99": float(np.nanpercentile(t_valid, 99)),
    "max": float(np.nanmax(t)),
    "mean": float(np.nanmean(t)),
    "std": float(np.nanstd(t)),
}
print("Resumo temperature_C (amostra):")
for k in [
    "n_total",
    "n_valid",
    "n_nan",
    "min",
    "p01",
    "p05",
    "p25",
    "median",
    "p75",
    "p95",
    "p99",
    "max",
    "mean",
    "std",
]:
    print(f"- {k}: {stats[k]}")

# Histograma (bins por Freedman–Diaconis com fallback)
q75, q25 = np.nanpercentile(t_valid, [75, 25])
iqr = q75 - q25
if iqr > 0:
    bin_w = 2 * iqr * (t_valid.shape[0] ** (-1 / 3))
    bins = int(np.clip(np.ceil((np.nanmax(t_valid) - np.nanmin(t_valid)) / max(bin_w, 1e-9)), 30, 160))
else:
    bins = 60

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.hist(t_valid, bins=bins, color="#4C72B0", edgecolor="#1f2d3d", linewidth=0.4)
ax.set_xlabel("temperature_C")
ax.set_ylabel("Frequência")
ax.set_title("Distribuição de temperature_C (amostra)")
ax.grid(True, axis="y", alpha=0.25)

msg = (
    f"n={stats['n_valid']:,}\\n"
    f"p05={stats['p05']:.2f} | p50={stats['median']:.2f} | p95={stats['p95']:.2f}\\n"
    f"mean={stats['mean']:.2f} ± {stats['std']:.2f}"
)
ax.text(
    0.98,
    0.98,
    msg,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.9),
)

fig.tight_layout()
fig.savefig(FIG_DIR / "eda_temperature_distribuicao.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_temperature_distribuicao.png")


Resumo temperature_C (amostra):
- n_total: 500000
- n_valid: 500000
- n_nan: 0
- min: 6.9
- p01: 13.0
- p05: 16.6
- p25: 23.5
- median: 26.2
- p75: 29.3
- p95: 35.5
- p99: 40.7
- max: 45.7
- mean: 26.203851799999995
- std: 5.401538401570126
Figura: /home/jovyan/work/notebooks/figuras/eda_temperature_distribuicao.png


## Relações feature–alvo (regressão em `temperature_C`)

Aqui a ideia é ver como a temperatura muda com:

- `hour` e `month` (sazonalidade)
- variáveis físicas diretamente relacionadas (`dew_point_C`, `humidity_pct`, `pressure_hPa`, etc.)

Vamos começar com **médias de `temperature_C` por `hour`/`month`** e depois um **scatter** com subsample para enxergar relação com `dew_point_C` e `humidity_pct`.


In [5]:
g_h = table.group_by("hour").aggregate([
    ("temperature_C", "mean"),
    ("temperature_C", "count"),
])
g_m = table.group_by("month").aggregate([
    ("temperature_C", "mean"),
    ("temperature_C", "count"),
])

# Ordenação (hour 0..23, month 1..12)
hour_vals = col_to_numpy(g_h.column("hour"), dtype=np.int64)
order_h = np.argsort(hour_vals)
h_x = hour_vals[order_h]
h_mean = col_to_numpy(g_h.column("temperature_C_mean"), dtype=np.float64)[order_h]
h_cnt = col_to_numpy(g_h.column("temperature_C_count"), dtype=np.float64)[order_h]

month_vals = col_to_numpy(g_m.column("month"), dtype=np.int64)
order_m = np.argsort(month_vals)
m_x = month_vals[order_m]
m_mean = col_to_numpy(g_m.column("temperature_C_mean"), dtype=np.float64)[order_m]
m_cnt = col_to_numpy(g_m.column("temperature_C_count"), dtype=np.float64)[order_m]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.plot(h_x, h_mean, color="#4C72B0", linewidth=1.8)
ax1.scatter(h_x, h_mean, s=22, color="#2b4c7e")
ax1.set_xlabel("hour")
ax1.set_ylabel("média temperature_C")
ax1.set_title("Média de temperature_C por hora (amostra)")
ax1.set_xticks(np.arange(0, 24, 2))
ax1.grid(True, axis="y", alpha=0.25)

ax2.plot(m_x, m_mean, color="#C44E52", linewidth=1.8)
ax2.scatter(m_x, m_mean, s=22, color="#7c2c2f")
ax2.set_xlabel("month")
ax2.set_ylabel("média temperature_C")
ax2.set_title("Média de temperature_C por mês (amostra)")
ax2.set_xticks(np.arange(1, 13, 1))
ax2.grid(True, axis="y", alpha=0.25)

fig.suptitle("Sazonalidade simples do alvo — médias condicionais", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_temperature_por_hour_month.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_temperature_por_hour_month.png")

print("Resumo group_by hour:")
print("hours:", h_x.tolist())
print("counts:", h_cnt.astype(int).tolist())
print("Resumo group_by month:")
print("months:", m_x.tolist())
print("counts:", m_cnt.astype(int).tolist())


Figura: /home/jovyan/work/notebooks/figuras/eda_temperature_por_hour_month.png
Resumo group_by hour:
hours: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
counts: [20834, 20834, 20834, 20834, 20834, 20834, 20834, 20834, 20833, 20833, 20833, 20833, 20833, 20833, 20833, 20833, 20833, 20833, 20833, 20833, 20833, 20833, 20833, 20833]
Resumo group_by month:
months: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
counts: [43152, 39336, 42840, 41040, 42408, 41040, 42408, 42408, 41040, 42344, 40320, 41664]


In [6]:
# Relação com variáveis físicas (subsample para visualização)
# Observação: dew_point_C é termodinamicamente ligado a temperature_C → correlação alta é esperada.

rng = np.random.default_rng(42)

temp = col_to_numpy(table.column("temperature_C"), dtype=np.float64)
dew = col_to_numpy(table.column("dew_point_C"), dtype=np.float64)
hum = col_to_numpy(table.column("humidity_pct"), dtype=np.float64)
pres = col_to_numpy(table.column("pressure_hPa"), dtype=np.float64)

valid = (~np.isnan(temp)) & (~np.isnan(dew)) & (~np.isnan(hum)) & (~np.isnan(pres))
idx = np.flatnonzero(valid)

n_plot = min(60_000, idx.shape[0])
if n_plot == 0:
    raise ValueError("Sem dados válidos suficientes para scatter (temperature/dew/humidity/pressure).")

pick = rng.choice(idx, size=n_plot, replace=False)

temp_s = temp[pick]
dew_s = dew[pick]
hum_s = hum[pick]
pres_s = pres[pick]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))

axes[0].scatter(dew_s, temp_s, s=6, alpha=0.22, color="#4C72B0", edgecolors="none")
axes[0].set_xlabel("dew_point_C")
axes[0].set_ylabel("temperature_C")
axes[0].set_title("temperature_C vs dew_point_C")
axes[0].grid(True, alpha=0.25)

axes[1].scatter(hum_s, temp_s, s=6, alpha=0.22, color="#55A868", edgecolors="none")
axes[1].set_xlabel("humidity_pct")
axes[1].set_ylabel("temperature_C")
axes[1].set_title("temperature_C vs humidity_pct")
axes[1].grid(True, alpha=0.25)

axes[2].scatter(pres_s, temp_s, s=6, alpha=0.22, color="#C44E52", edgecolors="none")
axes[2].set_xlabel("pressure_hPa")
axes[2].set_ylabel("temperature_C")
axes[2].set_title("temperature_C vs pressure_hPa")
axes[2].grid(True, alpha=0.25)

fig.suptitle("Relações alvo–preditores (subsample; amostra)", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_temperature_scatter_relacoes.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_temperature_scatter_relacoes.png")

# Correlações rápidas no subsample (sinal exploratório)
print("Corr(subsample) temperature vs dew_point:", float(np.corrcoef(temp_s, dew_s)[0, 1]))
print("Corr(subsample) temperature vs humidity:", float(np.corrcoef(temp_s, hum_s)[0, 1]))
print("Corr(subsample) temperature vs pressure:", float(np.corrcoef(temp_s, pres_s)[0, 1]))


Figura: /home/jovyan/work/notebooks/figuras/eda_temperature_scatter_relacoes.png
Corr(subsample) temperature vs dew_point: 0.14180477125202326
Corr(subsample) temperature vs humidity: -0.5408644517136181
Corr(subsample) temperature vs pressure: -0.3872627738451737


## Correlação entre preditores (amostra)

Subconjunto de colunas numéricas alinhadas a [preprocessamento.md](../docs/preprocessamento.md). A matriz usa `numpy.corrcoef`; valores ausentes são substituídos temporariamente pela média da coluna **só para este cálculo exploratório** (não substitui decisões do pipeline T022).


In [7]:
corr_cols = [
    "temperature_C",
    "humidity_pct",
    "precip_mm",
    "cloud_cover_pct",
    "pressure_hPa",
    "dew_point_C",
    "wind_speed_ms",
    "solar_radiation_Wm2",
]
rows = []
for c in corr_cols:
    v = col_to_numpy(table.column(c), dtype=np.float64)
    rows.append(v)
X = np.vstack(rows)
col_means = np.nanmean(X, axis=1, keepdims=True)
X_filled = np.where(np.isnan(X), col_means, X)
C = np.corrcoef(X_filled)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(C, vmin=-1, vmax=1, cmap="coolwarm", interpolation="nearest")
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)
ax.set_title("Correlação de Pearson — amostra; NaN imputados pela média da coluna")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_correlacao_preditoras.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_correlacao_preditoras.png")


Figura: /home/jovyan/work/notebooks/figuras/eda_correlacao_preditoras.png


In [8]:
# Ranking numérico das correlações (com base na matriz C já calculada)
# C e corr_cols vêm da célula anterior.

pairs = []
n = len(corr_cols)
for i in range(n):
    for j in range(i + 1, n):
        pairs.append((corr_cols[i], corr_cols[j], float(C[i, j])))

# Top por magnitude absoluta
pairs_abs = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)
# Top positivas e negativas
pairs_pos = sorted(pairs, key=lambda x: x[2], reverse=True)
pairs_neg = sorted(pairs, key=lambda x: x[2])

TOP_K = 10

print(f"Top {TOP_K} por |correlação|:")
for a, b, v in pairs_abs[:TOP_K]:
    print(f"- {a}  x  {b}: {v:+.4f}")

print(f"\nTop {TOP_K} correlações positivas:")
for a, b, v in pairs_pos[:TOP_K]:
    print(f"- {a}  x  {b}: {v:+.4f}")

print(f"\nTop {TOP_K} correlações negativas:")
for a, b, v in pairs_neg[:TOP_K]:
    print(f"- {a}  x  {b}: {v:+.4f}")

# Extra: ranking focado no alvo de regressão
target = "temperature_C"
with_target = [(a, b, v) for a, b, v in pairs if a == target or b == target]
with_target = sorted(with_target, key=lambda x: abs(x[2]), reverse=True)

print(f"\nCorrelações com {target} (ordenadas por |corr|):")
for a, b, v in with_target:
    other = b if a == target else a
    print(f"- {target}  x  {other}: {v:+.4f}")


Top 10 por |correlação|:
- humidity_pct  x  dew_point_C: +0.7212
- cloud_cover_pct  x  dew_point_C: +0.6296
- temperature_C  x  solar_radiation_Wm2: +0.5490
- temperature_C  x  humidity_pct: -0.5398
- humidity_pct  x  solar_radiation_Wm2: -0.4808
- humidity_pct  x  cloud_cover_pct: +0.4751
- temperature_C  x  pressure_hPa: -0.3821
- humidity_pct  x  pressure_hPa: +0.3760
- cloud_cover_pct  x  wind_speed_ms: +0.3076
- dew_point_C  x  wind_speed_ms: +0.2514

Top 10 correlações positivas:
- humidity_pct  x  dew_point_C: +0.7212
- cloud_cover_pct  x  dew_point_C: +0.6296
- temperature_C  x  solar_radiation_Wm2: +0.5490
- humidity_pct  x  cloud_cover_pct: +0.4751
- humidity_pct  x  pressure_hPa: +0.3760
- cloud_cover_pct  x  wind_speed_ms: +0.3076
- dew_point_C  x  wind_speed_ms: +0.2514
- precip_mm  x  cloud_cover_pct: +0.2326
- humidity_pct  x  precip_mm: +0.2092
- temperature_C  x  wind_speed_ms: +0.2075

Top 10 correlações negativas:
- temperature_C  x  humidity_pct: -0.5398
- humidity_

## Dimensão temporal (`hour`, `month`, `datetime`)

Exploração de sazonalidade horária e mensal e de contagem de registos por dia (data truncada a partir de `datetime`).

As contagens por dia usam **apenas as primeiras `N_AMOSTRA` linhas** do Parquet, na ordem em que o ficheiro foi lido; se essa fatia for temporalmente contígua e com cadência fixa por dia, o gráfico pode aparecer em **patamares** — isso reflecte a estrutura da amostra, não um erro de ordenação (os dias são ordenados antes do plot).


In [9]:
h_raw = col_to_numpy(table.column("hour"), dtype=np.float64)
m_raw = col_to_numpy(table.column("month"), dtype=np.float64)
h = h_raw[~np.isnan(h_raw)].astype(np.int64, copy=False)
m = m_raw[~np.isnan(m_raw)].astype(np.int64, copy=False)
h = np.clip(h, 0, 23)
m = np.clip(m, 1, 12)
c_h = np.bincount(h, minlength=24)
c_m = np.bincount(m - 1, minlength=12)
hours_x = np.arange(24)
months_x = np.arange(1, 13)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(hours_x, c_h, color="#55A868", edgecolor="#1a3d22", linewidth=0.65)
ax1.set_xlabel("hour (0–23)")
ax1.set_ylabel("Contagem")
ax1.set_title("Contagem por hora (amostra)")
ax1.set_xticks(np.arange(0, 24, 2))
ax1.grid(True, axis="y", alpha=0.3)

ax2.bar(months_x, c_m, color="#C44E52", edgecolor="#4a1518", linewidth=0.65)
ax2.set_xlabel("month (1–12)")
ax2.set_ylabel("Contagem")
ax2.set_title("Contagem por mês (amostra)")
ax2.set_xticks(months_x)
ax2.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_hour_month_histogramas.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_hour_month_histogramas.png")


Figura: /home/jovyan/work/notebooks/figuras/eda_hour_month_histogramas.png


In [10]:
d_arr = pc.cast(table.column("datetime"), pa.date32())
st = pc.value_counts(d_arr)
days = col_to_numpy(st.field(0), dtype=np.int64)
cnts = col_to_numpy(st.field(1), dtype=np.int64)
order = np.argsort(days)
days_s = days[order]
cnts_s = cnts[order].astype(float)
x = np.asarray(days_s, dtype="datetime64[D]")

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.step(x, cnts_s, where="post", color="#4C72B0", linewidth=1.1, label="contagem/dia")
ax.scatter(x, cnts_s, s=10, color="#2b4c7e", alpha=0.55, zorder=3)
ax.set_xlabel("Data")
ax.set_ylabel("Nº de linhas por dia (só na amostra)")
ax.set_title("Contagem por dia — fatia inicial de N linhas do Parquet")
ax.grid(True, axis="y", alpha=0.35)
ax.legend(loc="upper right", fontsize=8)
fig.autofmt_xdate()
fig.text(
    0.5,
    0.01,
    "Patamares = cadência estável por dia nesta fatia; dias ordenados antes do gráfico.",
    ha="center",
    fontsize=8,
    transform=fig.transFigure,
)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(FIG_DIR / "eda_contagem_por_dia.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_contagem_por_dia.png")


Figura: /home/jovyan/work/notebooks/figuras/eda_contagem_por_dia.png


## Figuras exportadas (caminhos)

Relativos à raiz do repositório:

- `notebooks/figuras/eda_temperature_distribuicao.png`
- `notebooks/figuras/eda_temperature_por_hour_month.png`
- `notebooks/figuras/eda_temperature_scatter_relacoes.png`
- `notebooks/figuras/eda_correlacao_preditoras.png`
- `notebooks/figuras/eda_hour_month_histogramas.png`
- `notebooks/figuras/eda_contagem_por_dia.png`

## Insights acionáveis (regressão em `temperature_C`)

1. **Faixa do alvo e outliers:** usar percentis (p05–p95) para entender caudas e evitar decisões baseadas em extremos raros. Se houver valores fisicamente improváveis, vale checar qualidade de dados antes de modelar.
2. **Sazonalidade:** `hour` e `month` devem explicar parte relevante do sinal; no pipeline, considerar representação **cíclica** (sin/cos) em vez de ordinal puro, e fazer **split temporal** para não “vazar” padrões.
3. **Preditores termodinâmicos:** `dew_point_C` tende a ser altamente informativo para `temperature_C`. Se o objetivo é previsão operacional, ok; se o objetivo é entender causalidade/impacto de outras variáveis, essa redundância pode “dominar” o modelo.
4. **Avaliação:** para regressão, preferir MAE/RMSE e métricas por fatias (por mês/hora/região) para garantir desempenho consistente ao longo do tempo e espaço.
5. **Próximo passo:** materializar splits e pipeline em Spark com **fit apenas no treino**, e validar generalização na janela temporal de validação/teste.
